# 05 — Fine-tune with progressive unfreezing and recovery checkpoints

This notebook trains the independent NVIDIA FastConformer PC v2 backbone, which is punctuation-aware but does not output diacritics. It uses the new immutable v2 manifest created in Notebook 01. Every 500 optimizer steps, the training state is saved as a Drive-backed recovery checkpoint. If Colab disconnects or the runtime is stopped, run the same training command again: the project detects the latest checkpoint and resumes the unfinished stage rather than starting that stage from zero.

The console is intentionally compact. It prints one stage header, a progress line every 100 steps, a checkpoint confirmation every 500 steps, and one validation summary. NeMo per-sample RNNT diagnostic predictions are suppressed because they are not the final CTC evaluation output.

In [ ]:
from pathlib import Path
import os

# Each notebook may open in a fresh Colab runtime, so mount Drive before any
# path check rather than relying on a previous notebook's session.
from google.colab import drive
DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.exists():
    drive.mount("/content/drive")

PROJECT_DIR = DRIVE_ROOT / "quran-fastconformer-colab"
assert PROJECT_DIR.exists(), f"Project directory not found: {PROJECT_DIR}"
os.chdir(PROJECT_DIR)
print("Working directory:", Path.cwd())


In [ ]:
from pathlib import Path
import importlib.util
import os
import shutil
import site
import subprocess
import sys

# Validate real imports, not merely package metadata. Recent Colab images can
# leave NumPy 2.x binary extensions behind after NeMo pins NumPy 1.26.4.
def _numpy_compatible_error():
    try:
        import numpy as np
        if np.__version__ != "1.26.4":
            return f"NumPy {np.__version__} is installed; this project requires 1.26.4"
        import pandas as pd
        from datasets import load_dataset  # noqa: F401
        return None
    except Exception as error:
        return f"binary/import compatibility check failed: {type(error).__name__}: {error}"


def _remove_stale_binary_packages():
    patterns = ("numpy", "numpy-*.dist-info", "pandas", "pandas-*.dist-info")
    for package_dir in site.getsitepackages():
        root = Path(package_dir)
        for pattern in patterns:
            for target in root.glob(pattern):
                if target.is_dir():
                    shutil.rmtree(target, ignore_errors=True)
                else:
                    target.unlink(missing_ok=True)


compatibility_error = _numpy_compatible_error()
required_modules = ("datasets", "jiwer", "soundfile", "yaml", "nemo", "matplotlib")
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]

if compatibility_error:
    print("Repairing incompatible NumPy/pandas binaries:", compatibility_error)
    subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "numpy", "pandas"])
    _remove_stale_binary_packages()
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "--force-reinstall",
        "numpy==1.26.4",
        "pandas==2.2.3",
    ])
    os.environ["QURAN_COLAB_RESTART_REQUIRED"] = "1"
    print("Clean NumPy repair completed. Use Runtime → Restart session before running any other cell.")
elif missing_modules:
    print("Installing missing project dependencies:", missing_modules)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir", "-r", "requirements.txt"])
    os.environ["QURAN_COLAB_RESTART_REQUIRED"] = "1"
    print("Dependencies updated. Use Runtime → Restart session before launching a NeMo stage.")
else:
    print("Core project dependencies and NumPy binary compatibility are available.")


In [ ]:
import os
import torch

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
assert torch.cuda.is_available(), "No GPU. Choose Runtime → Change runtime type → T4 GPU or L4 GPU."
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Inspect recovery state before training. This cell never changes the manifest.
import json
from pathlib import Path

state_path = Path("artifacts/experiments/fastconformer_pc_v2/nemo/training_state.json")
checkpoints = sorted(Path("artifacts/experiments/fastconformer_pc_v2/nemo/checkpoints").glob("**/last.ckpt"))
if state_path.exists():
    print(json.loads(state_path.read_text(encoding="utf-8")))
elif checkpoints:
    print("Recovery checkpoints found:")
    for path in checkpoints:
        print(" -", path)
else:
    print("No recovery checkpoint yet. The first run will start stage 1 from the pretrained model.")


In [ ]:
# This command is restart-safe. It resumes an unfinished stage from last.ckpt
# when one exists; completed stages are loaded from their stage_model.nemo export.
!python -m src.train --config configs/fastconformer_quran.yaml --manifest artifacts/experiments/fastconformer_pc_v2/manifests/experiment_manifest.json


In [ ]:
import json
from pathlib import Path

summary_path = Path("artifacts/experiments/fastconformer_pc_v2/models/training_summary.json")
state_path = Path("artifacts/experiments/fastconformer_pc_v2/nemo/training_state.json")
if summary_path.exists():
    print(json.loads(summary_path.read_text(encoding="utf-8")))
elif state_path.exists():
    print("Training is not complete yet. Current recovery state:")
    print(json.loads(state_path.read_text(encoding="utf-8")))
else:
    print("No completed training summary yet. Run the training cell first.")
